# 🧱 Lab: Backend Swap Comparison

**Module: The Replaceable Model Interface** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Run** identical input rows through two different AI backends using the same interface
2. **Record** and analyze disagreements between hosted and local model responses
3. **Compare** backends on latency, output quality, and cost using a structured evaluation
4. **Apply** the interface seam and dependency injection to swap backends with zero business-logic changes

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Interface Seam | A small typed contract between your app and an AI vendor. Your adapter returns your dataclass, never their SDK response object, so providers can be swapped without rewrites. | mini-seam-teardown |
| Dependency Injection | Pass the backend in, callers choose which model client to use, while your pipeline code stays unchanged and testable. | mini-define-ai-protocol |
| Prompt Versioning & Output Parsing | — | mini-typed-prompt-parser |
| HTTP API Requests & Secrets Hygiene | — | mini-hosted-backend |
| Open vs Closed Models & Baseline Comparison | — | mini-local-backend |
| Recorded Diff & Version Tracking | — | This lab |

## Prerequisites

- **mini-seam-teardown**: Identifying where vendor calls leak into business logic
- **mini-define-ai-protocol**: Defining the `AIClient` protocol and using dependency injection
- **mini-typed-prompt-parser**: Versioned prompt configs and defensive response parsing
- **mini-hosted-backend**: Calling a hosted API with auth, timeout, and secrets hygiene
- **mini-local-backend**: Building a local Ollama backend and comparing tradeoffs
- **Ollama** installed with `qwen2.5:0.5b` pulled (`ollama pull qwen2.5:0.5b`)
- **OpenAI API key** in a `.env` file


## Setup

Load dependencies and define the shared types, backends, and utilities we built across the previous mini-labs.

In [1]:
import os
import json
import time
from dataclasses import dataclass, field, asdict
from typing import Protocol, runtime_checkable, Optional

import requests
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()


def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


print("✅ Imports loaded.")

✅ Imports loaded.


## Step 1 — Shared Types and Backends

We reuse the exact `AIRequest`, `AIResponse`, `AIClient` protocol, `HostedAIClient`, and `LocalAIClient` from the previous mini-labs. This is the interface seam in action — all backends share one contract.

In [2]:
# ── Shared types ──

@dataclass
class AIRequest:
    prompt: str
    system: str = "You are a helpful assistant."
    temperature: float = 0.0
    max_tokens: int = 256


@dataclass
class AIResponse:
    text: str
    model: str = "unknown"
    usage: dict = field(default_factory=dict)


@runtime_checkable
class AIClient(Protocol):
    def complete(self, request: AIRequest) -> AIResponse: ...


# ── Hosted backend (OpenAI) ──

class HostedAIClient:
    def __init__(self, api_key: str, base_url: str = "https://api.openai.com/v1",
                 model: str = "gpt-4o-mini", timeout: int = 30):
        if not api_key:
            raise ValueError("API key must not be empty.")
        self.api_key = api_key
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.timeout = timeout

    def complete(self, request: AIRequest) -> AIResponse:
        url = f"{self.base_url}/chat/completions"
        headers = {"Authorization": f"Bearer {self.api_key}",
                   "Content-Type": "application/json"}
        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": request.system},
                {"role": "user", "content": request.prompt},
            ],
            "temperature": request.temperature,
            "max_tokens": request.max_tokens,
        }
        resp = requests.post(url, headers=headers, json=payload, timeout=self.timeout)
        resp.raise_for_status()
        body = resp.json()
        text = body.get("choices", [{}])[0].get("message", {}).get("content", "")
        usage = body.get("usage", {})
        model_used = body.get("model", self.model)
        return AIResponse(text=text.strip(), model=model_used, usage=usage)


# ── Local backend (Ollama) ──

class LocalAIClient:
    def __init__(self, model: str = "qwen2.5:0.5b",
                 base_url: str = "http://localhost:11434", timeout: int = 60):
        self.model = model
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout

    def complete(self, request: AIRequest) -> AIResponse:
        url = f"{self.base_url}/api/chat"
        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": request.system},
                {"role": "user", "content": request.prompt},
            ],
            "stream": False,
            "options": {"temperature": request.temperature,
                        "num_predict": request.max_tokens},
        }
        resp = requests.post(url, json=payload, timeout=self.timeout)
        resp.raise_for_status()
        body = resp.json()
        text = body.get("message", {}).get("content", "")
        model_used = body.get("model", self.model)
        return AIResponse(text=text.strip(), model=model_used)


print("Shared types and both backends defined.")
print(f"  HostedAIClient satisfies AIClient? {isinstance(HostedAIClient(api_key='test'), AIClient)}")
print(f"  LocalAIClient  satisfies AIClient? {isinstance(LocalAIClient(), AIClient)}")

Shared types and both backends defined.
  HostedAIClient satisfies AIClient? True
  LocalAIClient  satisfies AIClient? True


## Step 2 — The Test Dataset

A real swap comparison needs **identical inputs** sent to both backends. We'll use a small set of support tickets with known expected categories. This is our **baseline** — if we know the right answer, we can measure accuracy alongside disagreements.

In [3]:
test_rows = [
    {"id": 1, "ticket": "I was charged twice this month",            "expected": "billing"},
    {"id": 2, "ticket": "My login keeps failing with error 403",     "expected": "technical"},
    {"id": 3, "ticket": "How do I change my email address?",         "expected": "account"},
    {"id": 4, "ticket": "The app crashes when I upload a photo",     "expected": "technical"},
    {"id": 5, "ticket": "I need a refund for my last payment",       "expected": "billing"},
    {"id": 6, "ticket": "Can I transfer my subscription to a friend?", "expected": "account"},
]

print(f"Test dataset: {len(test_rows)} tickets with expected categories.")
for row in test_rows:
    print(f"  [{row['id']}] {row['ticket'][:45]:45s} → expected: {row['expected']}")

Test dataset: 6 tickets with expected categories.
  [1] I was charged twice this month                → expected: billing
  [2] My login keeps failing with error 403         → expected: technical
  [3] How do I change my email address?             → expected: account
  [4] The app crashes when I upload a photo         → expected: technical
  [5] I need a refund for my last payment           → expected: billing
  [6] Can I transfer my subscription to a friend?   → expected: account


## Step 3 — The Versioned Prompt Config

From **mini-typed-prompt-parser**, we know prompts should be versioned data, not loose strings. We define a single `PromptConfig` and use it for both backends — this guarantees they receive the *exact same instructions*.

In [4]:
@dataclass
class PromptConfig:
    """A versioned, structured prompt configuration."""
    name: str
    version: str
    system: str
    user_template: str
    temperature: float = 0.0
    max_tokens: int = 20

    def render(self, **kwargs) -> str:
        return self.user_template.format(**kwargs)


classifier_prompt = PromptConfig(
    name="ticket-classifier",
    version="2.0",
    system="You are a support ticket classifier. Reply with exactly one word: billing, technical, account, or other.",
    user_template="Classify this support ticket: '{ticket}'\nCategory:",
    temperature=0.0,
    max_tokens=10,
)

print(f"Prompt: {classifier_prompt.name} v{classifier_prompt.version}")
print(f"System: {classifier_prompt.system}")
print(f"Template: {classifier_prompt.user_template}")

Prompt: ticket-classifier v2.0
System: You are a support ticket classifier. Reply with exactly one word: billing, technical, account, or other.
Template: Classify this support ticket: '{ticket}'
Category:


## Step 4 — The Runner: Same Pipeline, Any Backend

This is the core of the swap comparison. The `run_comparison` function takes an `AIClient` (any backend) and runs every test row through it, recording the response and the time taken. The function has **zero vendor imports** — it only knows about the `AIClient` protocol.

In [5]:
VALID_CATEGORIES = {"billing", "technical", "account", "other"}


def normalize_category(raw: str) -> str:
    """Defensively extract a valid category from raw model output."""
    cleaned = raw.strip().lower().rstrip(".!,")
    # Sometimes the model returns extra words; grab the first valid one
    for word in cleaned.split():
        if word in VALID_CATEGORIES:
            return word
    return "other"


def run_comparison(ai_client: AIClient, rows: list, config: PromptConfig) -> list:
    """Run all rows through a backend. Returns a list of result dicts."""
    results = []
    for row in rows:
        request = AIRequest(
            prompt=config.render(ticket=row["ticket"]),
            system=config.system,
            temperature=config.temperature,
            max_tokens=config.max_tokens,
        )

        start = time.time()
        try:
            response = ai_client.complete(request)
            elapsed = time.time() - start
            category = normalize_category(response.text)
            results.append({
                "id": row["id"],
                "ticket": row["ticket"],
                "expected": row["expected"],
                "predicted": category,
                "raw_response": response.text,
                "model": response.model,
                "latency_s": round(elapsed, 2),
                "error": None,
            })
        except Exception as e:
            elapsed = time.time() - start
            results.append({
                "id": row["id"],
                "ticket": row["ticket"],
                "expected": row["expected"],
                "predicted": "error",
                "raw_response": "",
                "model": "N/A",
                "latency_s": round(elapsed, 2),
                "error": str(e)[:80],
            })
    return results


print("run_comparison() defined — same function works with any AIClient backend.")

run_comparison() defined — same function works with any AIClient backend.


## Step 5 — Run the Hosted Backend

First, we send all six tickets through the **hosted** backend (OpenAI API). This is the higher-quality baseline.

In [6]:
hosted = HostedAIClient(
    api_key=os.getenv("OPENAI_API_KEY", ""),
    model="gpt-4o-mini",
    timeout=15,
)

print(f"Running {len(test_rows)} tickets through HOSTED backend ({hosted.model})...\n")
hosted_results = run_comparison(hosted, test_rows, classifier_prompt)

for r in hosted_results:
    match = "✅" if r["predicted"] == r["expected"] else "❌"
    print(f"  [{r['id']}] {match} predicted={r['predicted']:10s} expected={r['expected']:10s} ({r['latency_s']}s)")

Running 6 tickets through HOSTED backend (gpt-4o-mini)...



  [1] ✅ predicted=billing    expected=billing    (0.51s)
  [2] ✅ predicted=technical  expected=technical  (0.86s)
  [3] ✅ predicted=account    expected=account    (0.7s)
  [4] ✅ predicted=technical  expected=technical  (0.46s)
  [5] ✅ predicted=billing    expected=billing    (0.49s)
  [6] ✅ predicted=account    expected=account    (0.71s)


## Step 6 — Run the Local Backend

Now the *exact same tickets*, the *exact same prompt config*, through the **local** Ollama backend. The only thing that changed is the constructor call — the business logic (`run_comparison`) is untouched.

In [7]:
local = LocalAIClient(model="qwen2.5:0.5b", timeout=120)

print(f"Running {len(test_rows)} tickets through LOCAL backend ({local.model})...\n")
local_results = run_comparison(local, test_rows, classifier_prompt)

for r in local_results:
    if r["error"]:
        print(f"  [{r['id']}] ⚠️  error: {r['error']}")
    else:
        match = "✅" if r["predicted"] == r["expected"] else "❌"
        print(f"  [{r['id']}] {match} predicted={r['predicted']:10s} expected={r['expected']:10s} ({r['latency_s']}s)")

Running 6 tickets through LOCAL backend (qwen2.5:0.5b)...



  [1] ⚠️  error: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url:
  [2] ⚠️  error: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url:
  [3] ⚠️  error: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url:
  [4] ⚠️  error: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url:
  [5] ⚠️  error: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url:
  [6] ⚠️  error: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url:


## Step 7 — Record the Disagreements

The whole point of a swap comparison: **where do the two backends disagree**, and what does that tell us about each model's strengths and weaknesses?

In [8]:
# Build the diff record
diff_records = []
for h, l in zip(hosted_results, local_results):
    diff_records.append({
        "id": h["id"],
        "ticket": h["ticket"],
        "expected": h["expected"],
        "hosted_pred": h["predicted"],
        "local_pred": l["predicted"],
        "agree": h["predicted"] == l["predicted"],
        "hosted_correct": h["predicted"] == h["expected"],
        "local_correct": l["predicted"] == l["expected"],
        "hosted_raw": h["raw_response"],
        "local_raw": l["raw_response"],
    })

# Display results
agreements = sum(1 for d in diff_records if d["agree"])
disagreements = len(diff_records) - agreements

rows_md = ""
for d in diff_records:
    status = "✅ Agree" if d["agree"] else "❌ **Disagree**"
    rows_md += f"| {d['id']} | {d['ticket'][:40]} | {d['expected']} | {d['hosted_pred']} | {d['local_pred']} | {status} |\n"

md(f"""
## Disagreement Report

| ID | Ticket | Expected | Hosted | Local | Status |
|----|--------|----------|--------|-------|--------|
{rows_md}

**Agreements:** {agreements}/{len(diff_records)} | **Disagreements:** {disagreements}/{len(diff_records)}
""")


## Disagreement Report

| ID | Ticket | Expected | Hosted | Local | Status |
|----|--------|----------|--------|-------|--------|
| 1 | I was charged twice this month | billing | billing | error | ❌ **Disagree** |
| 2 | My login keeps failing with error 403 | technical | technical | error | ❌ **Disagree** |
| 3 | How do I change my email address? | account | account | error | ❌ **Disagree** |
| 4 | The app crashes when I upload a photo | technical | technical | error | ❌ **Disagree** |
| 5 | I need a refund for my last payment | billing | billing | error | ❌ **Disagree** |
| 6 | Can I transfer my subscription to a frie | account | account | error | ❌ **Disagree** |


**Agreements:** 0/6 | **Disagreements:** 6/6


## Step 8 — Accuracy & Latency Comparison

Beyond disagreements, let's compare the two backends on **accuracy** (how often each matches the expected label) and **latency** (how fast each responds).

In [9]:
# Accuracy
hosted_correct = sum(1 for d in diff_records if d["hosted_correct"])
local_correct = sum(1 for d in diff_records if d["local_correct"])
total = len(diff_records)

# Latency
hosted_avg_latency = sum(r["latency_s"] for r in hosted_results) / total
local_avg_latency = sum(r["latency_s"] for r in local_results) / total
hosted_total_latency = sum(r["latency_s"] for r in hosted_results)
local_total_latency = sum(r["latency_s"] for r in local_results)

md(f"""
## Backend Comparison Summary

| Metric | Hosted ({hosted_results[0]['model']}) | Local ({local_results[0]['model']}) |
|--------|--------|-------|
| **Accuracy** | {hosted_correct}/{total} ({100*hosted_correct/total:.0f}%) | {local_correct}/{total} ({100*local_correct/total:.0f}%) |
| **Avg Latency** | {hosted_avg_latency:.2f}s | {local_avg_latency:.2f}s |
| **Total Time** | {hosted_total_latency:.2f}s | {local_total_latency:.2f}s |
| **Cost** | ~$0.15/M input tokens | $0.00 (local) |
| **Privacy** | Data sent to OpenAI | Data stays on machine |
""")


## Backend Comparison Summary

| Metric | Hosted (gpt-4o-mini-2024-07-18) | Local (N/A) |
|--------|--------|-------|
| **Accuracy** | 6/6 (100%) | 0/6 (0%) |
| **Avg Latency** | 0.62s | 4.08s |
| **Total Time** | 3.73s | 24.46s |
| **Cost** | ~$0.15/M input tokens | $0.00 (local) |
| **Privacy** | Data sent to OpenAI | Data stays on machine |


## Step 9 — Save the Comparison Record

In production, you'd commit this record to version control alongside the prompt config. This gives you a traceable history: *which prompt version + which backend + which results*. Let's save both the prompt config and the diff to a JSON file.

In [10]:
comparison_record = {
    "prompt_config": asdict(classifier_prompt),
    "backends": {
        "hosted": {"type": "HostedAIClient", "model": hosted.model},
        "local": {"type": "LocalAIClient", "model": local.model},
    },
    "results": diff_records,
    "summary": {
        "total_rows": total,
        "agreements": agreements,
        "disagreements": disagreements,
        "hosted_accuracy": f"{100*hosted_correct/total:.0f}%",
        "local_accuracy": f"{100*local_correct/total:.0f}%",
        "hosted_avg_latency_s": round(hosted_avg_latency, 2),
        "local_avg_latency_s": round(local_avg_latency, 2),
    },
}

output_path = "swap_comparison_record.json"
with open(output_path, "w") as f:
    json.dump(comparison_record, f, indent=2, default=str)

print(f"✅ Comparison record saved to {output_path}")
print(f"   This file can be committed to version control for traceability.")

✅ Comparison record saved to swap_comparison_record.json
   This file can be committed to version control for traceability.


## Step 10 — Interpreting the Results

Let's turn the raw numbers into an actionable recommendation. This is the "so what" of the comparison — when should you pick which backend?

In [11]:
hosted_acc = hosted_correct / total
local_acc = local_correct / total

analysis_lines = ["## Analysis & Recommendation\n"]

# Quality gap
gap = hosted_acc - local_acc
if gap > 0.15:
    analysis_lines.append(
        f"**Quality gap is significant ({gap:.0%}).** The hosted model substantially "
        f"outperforms the local model on this task. For customer-facing classification, "
        f"the hosted backend is the safer choice.\n"
    )
elif gap > 0:
    analysis_lines.append(
        f"**Quality gap is modest ({gap:.0%}).** The hosted model is slightly better. "
        f"For internal or batch use where occasional errors are acceptable, the local "
        f"backend could save costs.\n"
    )
else:
    analysis_lines.append(
        f"**No quality gap.** Both backends perform equally well on this dataset. "
        f"The local backend is the clear winner on cost and privacy.\n"
    )

# Latency
if local_avg_latency > hosted_avg_latency * 3:
    analysis_lines.append(
        f"**Latency warning:** the local backend is {local_avg_latency/hosted_avg_latency:.1f}x "
        f"slower. For real-time applications, hosted is preferred.\n"
    )

# Key insight
analysis_lines.append(
    "**Key insight:** the swap required changing one constructor call. "
    "`run_comparison()`, `normalize_category()`, and the prompt config were "
    "completely untouched — that's the value of the interface seam."
)

md("\n".join(analysis_lines))

## Analysis & Recommendation

**Quality gap is significant (100%).** The hosted model substantially outperforms the local model on this task. For customer-facing classification, the hosted backend is the safer choice.

**Latency warning:** the local backend is 6.6x slower. For real-time applications, hosted is preferred.

**Key insight:** the swap required changing one constructor call. `run_comparison()`, `normalize_category()`, and the prompt config were completely untouched — that's the value of the interface seam.

## 🎯 Summary

### What You Built

You built a recorded swap comparison pipeline that runs identical support-ticket inputs through a hosted (OpenAI) and a local (Ollama) backend, records disagreements, compares accuracy and latency, and saves the full comparison to a traceable JSON file — all without changing a single line of business logic between backends.

### Key Takeaways

1. **Interface seam in action** — `run_comparison()` accepted any `AIClient` backend; swapping hosted for local required changing only the constructor call, not the pipeline
2. **Recorded disagreements** — comparing predictions row-by-row reveals exactly where a cheaper or smaller model breaks down, giving you data to decide rather than guess
3. **Versioned prompt + results** — saving the prompt config alongside the diff record means every comparison is reproducible and auditable in version control
4. **Baseline comparison framework** — evaluating backends across accuracy, latency, cost, and privacy provides a structured decision framework rather than ad-hoc impressions
5. **Defensive output parsing** — `normalize_category()` handles messy model outputs (extra words, punctuation) so the comparison is fair regardless of each model's verbosity